# Group-Average CF Modeling — Results Visualization

Pycortex flatmaps for CF model outputs from scripts 03–06.
Use wb_view for CIFTI visualization (files in `cifti_maps/`).

**Before running:** confirm `cortex.database.default_filestore` points to your HCP filestore
and that subject `hcp_999999_draw_NH` is registered.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cortex

# Make shared/ and vendor/ importable
CF_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if CF_DIR not in sys.path:
    sys.path.insert(0, CF_DIR)

from shared.vis_utils import basic_plot, alpha_plot, Plot

In [ ]:
# =============================================================================
# CONFIG — edit these
# =============================================================================
ROI_A = "A1"
ROI_B = "V1"

OUTPUT_BASE = "/home/amin/Research/Representation/Movie/outputs/cf_modeling/group_average"
HCP_DIR     = "/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1"
CX_SUB      = "hcp_999999_draw_NH"   # pycortex subject name

OUTPUT_DIR  = f"{OUTPUT_BASE}/{ROI_A}_{ROI_B}"
CIFTI_DIR   = f"{OUTPUT_DIR}/cifti_maps"
RESULTS_DIR = f"{OUTPUT_DIR}/results"
FIGURES_DIR = f"{OUTPUT_DIR}/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

cortex.database.default_filestore = HCP_DIR
print(f"ROI pair: {ROI_A} × {ROI_B}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Load all maps (59412 vertices, 32k_fs_LR)
R2_full = np.load(f"{OUTPUT_DIR}/R2_full.npy")
R2_a    = np.load(f"{OUTPUT_DIR}/R2_{ROI_A}.npy")
R2_b    = np.load(f"{OUTPUT_DIR}/R2_{ROI_B}.npy")
shared  = np.load(f"{OUTPUT_DIR}/Shared_R2.npy")

R2_a_nc       = np.load(f"{OUTPUT_DIR}/R2_{ROI_A}_nc.npy")
R2_b_nc       = np.load(f"{OUTPUT_DIR}/R2_{ROI_B}_nc.npy")
R2_null_a     = np.load(f"{OUTPUT_DIR}/R2_null_{ROI_A}.npy")
R2_null_b     = np.load(f"{OUTPUT_DIR}/R2_null_{ROI_B}.npy")

integration   = np.load(f"{OUTPUT_DIR}/integration_score.npy")
balance       = np.load(f"{OUTPUT_DIR}/modality_balance.npy")

print(f"R2_full: mean={R2_full.mean():.4f}  frac>0.01={np.mean(R2_full>0.01):.1%}")
print(f"R2_{ROI_A}_nc: mean={R2_a_nc.mean():.4f}  frac>0={np.mean(R2_a_nc>0):.1%}")
print(f"R2_{ROI_B}_nc: mean={R2_b_nc.mean():.4f}  frac>0={np.mean(R2_b_nc>0):.1%}")
print(f"integration: mean={integration.mean():.5f}  frac>0={np.mean(integration>0):.1%}")

## 1. Full-model R²

In [ ]:
vmax_full = np.percentile(R2_full[R2_full > 0], 95) if (R2_full > 0).any() else 0.1
fig_full = basic_plot(R2_full, vmax=vmax_full, subject=CX_SUB, vmin=0, cmap="viridis")
fig_full.suptitle(f"Full model R² (banded ridge: {ROI_A} + {ROI_B})")

## 2. Variance partitioning — individual band R²

In [ ]:
vmax_a = np.percentile(R2_a[R2_a > 0], 95) if (R2_a > 0).any() else 0.1
vmax_b = np.percentile(R2_b[R2_b > 0], 95) if (R2_b > 0).any() else 0.1

fig_a = basic_plot(R2_a, vmax=vmax_a, subject=CX_SUB, vmin=0, cmap="Reds")
fig_a.suptitle(f"Split R² — {ROI_A} band")

fig_b = basic_plot(R2_b, vmax=vmax_b, subject=CX_SUB, vmin=0, cmap="Blues")
fig_b.suptitle(f"Split R² — {ROI_B} band")

## 3. Null-corrected R² maps

In [ ]:
vmax_a_nc = np.percentile(R2_a_nc[R2_a_nc > 0], 95) if (R2_a_nc > 0).any() else 0.1
vmax_b_nc = np.percentile(R2_b_nc[R2_b_nc > 0], 95) if (R2_b_nc > 0).any() else 0.1

fig_anc = basic_plot(R2_a_nc, vmax=vmax_a_nc, subject=CX_SUB, vmin=0, cmap="Reds")
fig_anc.suptitle(f"Null-corrected R² — {ROI_A}")

fig_bnc = basic_plot(R2_b_nc, vmax=vmax_b_nc, subject=CX_SUB, vmin=0, cmap="Blues")
fig_bnc.suptitle(f"Null-corrected R² — {ROI_B}")

## 4. 2D variance decomposition — Figure 3a equivalent

Red = {ROI_A} only, Blue = {ROI_B} only, Purple = bimodal integration zone

In [ ]:
p = Plot(
    dat=R2_a_nc,
    dat2=R2_b_nc,
    vmin=0, vmax=vmax_a_nc,
    vmin2=0, vmax2=vmax_b_nc,
    subject=CX_SUB,
    with_curvature=True,
    with_rois=True,
)
p.uber_plot()
p.fig.suptitle(f"Variance decomposition: {ROI_A} (red) × {ROI_B} (blue)  [Figure 3a]")

out_path = f"{FIGURES_DIR}/variance_decomp_2D.png"
p.saveout(outloc=out_path)
print(f"Saved: {out_path}")

## 5. Integration score √(R²_A × R²_B)

In [ ]:
vmax_int = np.percentile(integration[integration > 0], 95) if (integration > 0).any() else 0.01
fig_int = basic_plot(integration, vmax=vmax_int, subject=CX_SUB, vmin=0, cmap="viridis")
fig_int.suptitle(f"Integration score √(R²_{ROI_A} × R²_{ROI_B})")

## 6. Modality balance (R²_A − R²_B)

In [ ]:
vmax_bal = np.percentile(np.abs(balance), 95)
fig_bal = basic_plot(balance, vmax=vmax_bal, subject=CX_SUB, vmin=-vmax_bal, cmap="RdBu_r")
fig_bal.suptitle(f"Modality balance: R²_{ROI_A} − R²_{ROI_B}  (red={ROI_A}, blue={ROI_B})")

## 7. RSA overlap maps (script 06)

In [ ]:
# Load overlap scores for all available configs
overlap_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("overlap_score_") and f.endswith(".npy")]
print(f"Found {len(overlap_files)} overlap maps:")
for fn in sorted(overlap_files):
    arr = np.load(f"{OUTPUT_DIR}/{fn}")
    print(f"  {fn}: mean={arr.mean():.4f}  frac>0.1={np.mean(arr>0.1):.1%}")

In [ ]:
# Visualize overlap for the primary config
PRIMARY_CONFIG = "notnormalized_nohrf_global_2s"
overlap_path = f"{OUTPUT_DIR}/overlap_score_{PRIMARY_CONFIG}.npy"

if os.path.exists(overlap_path):
    overlap = np.load(overlap_path)
    vmax_ov = np.percentile(overlap[overlap > 0], 95) if (overlap > 0).any() else 0.5
    fig_ov = basic_plot(overlap, vmax=vmax_ov, subject=CX_SUB, vmin=0, cmap="PuRd")
    fig_ov.suptitle(f"CF × RSA overlap ({PRIMARY_CONFIG})")
else:
    print(f"Not found: {overlap_path}  — run script 06 first")

## 8. RSA overlap correlation summary

In [ ]:
json_files = [f for f in os.listdir(RESULTS_DIR) if f.startswith("rsa_overlap_") and f.endswith(".json")]
rows = []
for fn in sorted(json_files):
    with open(f"{RESULTS_DIR}/{fn}") as fh:
        d = json.load(fh)
    rows.append({
        "config":  d.get("config", fn),
        "rho_joint": d.get("rho_integration_vs_rsa_joint", float("nan")),
        "ci_lo":   d.get("ci_lo_integration_vs_rsa_joint", float("nan")),
        "ci_hi":   d.get("ci_hi_integration_vs_rsa_joint", float("nan")),
    })

if rows:
    print(f"{'Config':<50}  {'rho':>7}  {'95% CI':>16}")
    print("-" * 78)
    for r in rows:
        print(f"{r['config']:<50}  {r['rho_joint']:>7.4f}  [{r['ci_lo']:>7.4f}, {r['ci_hi']:>7.4f}]")
else:
    print("No result JSON files found — run script 06 first")

## 9. Save flatmaps to PNG

In [ ]:
def save_flatmap(data, vmax, cmap, name, vmin=0, subject=CX_SUB):
    vx  = cortex.Vertex(data, subject=subject, vmin=vmin, vmax=vmax, cmap=cmap)
    fig = cortex.quickflat.make_figure(vx, with_curvature=True, with_rois=True)
    out = f"{FIGURES_DIR}/{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")

save_flatmap(R2_a_nc,     vmax_a_nc, "Reds",    f"R2_{ROI_A}_nc")
save_flatmap(R2_b_nc,     vmax_b_nc, "Blues",   f"R2_{ROI_B}_nc")
save_flatmap(integration, vmax_int,  "viridis", "integration_score")
save_flatmap(balance,     vmax_bal,  "RdBu_r",  "modality_balance", vmin=-vmax_bal)

if os.path.exists(overlap_path):
    save_flatmap(overlap, vmax_ov, "PuRd", f"overlap_score_{PRIMARY_CONFIG}")

---
## wb_view: CIFTI visualization

Use wb_view for vertex-level inspection of the 32k CIFTI files:

```bash
CIFTI_DIR="<output_base>/A1_V1/cifti_maps"

# Continuous overlays (Figure 3a — load both, set min transparent)
wb_view $CIFTI_DIR/R2_A1_nc.dscalar.nii  $CIFTI_DIR/R2_V1_nc.dscalar.nii

# Discrete 4-color label map
wb_view $CIFTI_DIR/bimodal_map.dlabel.nii

# Integration score
wb_view $CIFTI_DIR/integration_score.dscalar.nii

# RSA overlap
wb_view $CIFTI_DIR/overlap_score_notnormalized_nohrf_global_2s.dscalar.nii
```